# Tensor如何节省内存

运行一些计算操作的时候，例如之前的加减乘除分为*不修改原张量*，和*修改原张量*，不修改原张量就会导致在内存新开辟一个空间存放这些数据，造成资源的浪费，如果推广到高维张量运算，例如`X=X@Y`,毫无疑问计算结果会新开辟一段内存存储新的运算结果，有没有什么方法可以节省内存，让新的运算结果存储到`X`中呢？

- 注意：`X[:] = X @ Y` 要求右侧结果的形状和 `X` 原来的形状一致。如果矩阵乘法后的形状发生改变，不能直接写回 `X[:]`。

In [1]:
import torch


def show_tensor_info(name, tensor):
    print(f"{name}:")
    print(tensor)
    print("shape:", tensor.shape)
    print("Python对象id:", id(tensor))
    print("底层数据地址:", tensor.data_ptr())
    print()


# 1. 普通写法：X = X + Y 会创建新的 Tensor 对象和新的底层存储
X = torch.ones(2, 3)
Y = torch.full((2, 3), 2.0)

show_tensor_info("原始 X", X)
old_id = id(X)
old_data_ptr = X.data_ptr()

X = X + Y

show_tensor_info("X = X + Y 之后", X)
print("Python对象是否相同:", id(X) == old_id)
print("底层数据地址是否相同:", X.data_ptr() == old_data_ptr)
print("-" * 50)


# 2. 切片写回：X[:] = X + Y 会把结果写回 X 原来的内存位置
X = torch.ones(2, 3)
Y = torch.full((2, 3), 2.0)

show_tensor_info("原始 X", X)
old_id = id(X)
old_data_ptr = X.data_ptr()

X[:] = X + Y

show_tensor_info("X[:] = X + Y 之后", X)
print("Python对象是否相同:", id(X) == old_id)
print("底层数据地址是否相同:", X.data_ptr() == old_data_ptr)
print("-" * 50)


# 3. 矩阵乘法写回：只有当 X @ Y 的结果形状和 X 原形状相同时才可以
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
Y = torch.tensor([[1.0, 0.0], [0.0, 1.0]])

show_tensor_info("原始 X", X)
old_id = id(X)
old_data_ptr = X.data_ptr()

X[:] = X @ Y

show_tensor_info("X[:] = X @ Y 之后", X)
print("Python对象是否相同:", id(X) == old_id)
print("底层数据地址是否相同:", X.data_ptr() == old_data_ptr)
print("-" * 50)


# 4. 如果矩阵乘法结果形状变了，就不能直接写回 X[:]
X = torch.ones(2, 3)
Y = torch.ones(3, 4)

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("(X @ Y).shape:", (X @ Y).shape)

try:
    X[:] = X @ Y
except RuntimeError as error:
    print("形状不一致，不能写回 X[:]：")
    print(error)

原始 X:
tensor([[1., 1., 1.],
        [1., 1., 1.]])
shape: torch.Size([2, 3])
Python对象id: 130347254921264
底层数据地址: 243649536

X = X + Y 之后:
tensor([[3., 3., 3.],
        [3., 3., 3.]])
shape: torch.Size([2, 3])
Python对象id: 130343149752048
底层数据地址: 197574784

Python对象是否相同: False
底层数据地址是否相同: False
--------------------------------------------------
原始 X:
tensor([[1., 1., 1.],
        [1., 1., 1.]])
shape: torch.Size([2, 3])
Python对象id: 130343146804608
底层数据地址: 243747520

X[:] = X + Y 之后:
tensor([[3., 3., 3.],
        [3., 3., 3.]])
shape: torch.Size([2, 3])
Python对象id: 130343146804608
底层数据地址: 243747520

Python对象是否相同: True
底层数据地址是否相同: True
--------------------------------------------------
原始 X:
tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])
Python对象id: 130343148819680
底层数据地址: 243729280

X[:] = X @ Y 之后:
tensor([[1., 2.],
        [3., 4.]])
shape: torch.Size([2, 2])
Python对象id: 130343148819680
底层数据地址: 243729280

Python对象是否相同: True
底层数据地址是否相同: True
------------------------------